# class-label-sources — a stronger YOLOE on the Gonchary orthophoto

Probe 4 ran `yoloe-11l-seg` over Esri z18 village tiles at 0.386 m/px, tiled at
1024 px, and came back `failed`. Two things are different here:

- **A stronger checkpoint.** `ultralytics 8.4.150` ships a newer YOLOE
  generation — `yoloe-26n/s/m/l/x-seg`. This runs **`yoloe-26x-seg`** (163 MB,
  MobileCLIP2-B text encoder) against 11l's 71 MB / MobileCLIP-BLT.
- **Smaller patches on better imagery.** The 2025 Gonchary Google Earth export
  is 8192x6026 over roughly 545 x 406 m — **6.6 cm/px**, about 6x the ground
  resolution of the z18 tiles. A 50 m patch is 750 px native, so a 10 m building
  is ~150 px wide in frame instead of 26.

The raw `satellite_gonchary.jpg` is used, not the `_corrected` / `_cutted`
variants: those are warped into the flight-area box and are 50% black, with the
village masked out — no buildings left to find.

Classes are the four non-background ones we have, with probe 4's phrasing —
bare nouns return zero masks, the `"... in satellite image"` suffix is what
makes the text encoder fire.

**No verdict is written here.** This notebook runs the model and draws what it
returns. Nothing is scored against a control and nothing is concluded.

In [ ]:
# papermill parameters
variant   = "yoloe-26x-seg.pt"   # probe 4 used yoloe-11l-seg.pt
ortho     = "satellite_gonchary.jpg"   # raw Google Earth export; the _corrected
                                       # and _cutted variants are 50% black
patch_m   = 50                   # ground extent of one patch, metres
imgsz     = 640                  # what each patch is resized to before inference
conf      = 0.05                 # same floor as probe 4
sizes     = [25, 50, 100, 200]   # extents for the size comparison at the end
n_sample  = 12                   # patches drawn in the sample grid

## Setup

In [ ]:
import json, time, warnings
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Patch
from PIL import Image

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None

HERE    = Path.cwd()
SRC     = HERE.parents[1]                       # .../src
ORTHO   = SRC.parent / "archive/uav_localization_2025/data/satelites" / ortho
FIGURES = HERE / "figures"
FIGURES.mkdir(exist_ok=True)

# The corners of the *corrected* export, from the archive's
# gps_to_satellite_point.ipynb. Copied rather than imported: pocs/map-derived-labels
# owns geo.py and PoCs do not import from each other. The raw export covers the
# same scene at the same pixel size, so the metres-per-pixel below is nominal --
# enough to set a patch's ground extent, not a georeference.
TOP, LEFT, BOTTOM, RIGHT = 49.762965, 24.166472, 49.759291, 24.174046

CLASSES = ["building", "track", "woody", "water"]
PROMPTS = ["building in satellite image", "dirt road in satellite image",
           "trees in satellite image", "water in satellite image"]
COLOUR  = {"building": "#e6194b", "track": "#f58231",
           "woody": "#3cb44b", "water": "#4363d8"}
print(f"ortho exists: {ORTHO.exists()}")

## The imagery

One 2025 Google Earth export over the flight ground: village, road, treelines
and open field in the same frame.

In [ ]:
image = Image.open(ORTHO).convert("RGB")
image.load()
W, H = image.size

lat_mid = (TOP + BOTTOM) / 2
width_m  = (RIGHT - LEFT) * 111_320 * np.cos(np.radians(lat_mid))
height_m = (TOP - BOTTOM) * 110_574
mpp = width_m / W

print(f"{ORTHO.name}")
print(f"  {W} x {H} px   {width_m:.0f} x {height_m:.0f} m   {mpp*100:.2f} cm/px")
print(f"  a {patch_m} m patch is {patch_m/mpp:.0f} px native, resized to {imgsz}")

fig, ax = plt.subplots(figsize=(11, 8))
ax.imshow(image.resize((W // 6, H // 6)))
ax.set_title(f"Gonchary — {width_m:.0f} x {height_m:.0f} m at {mpp*100:.1f} cm/px")
ax.axis("off")
plt.tight_layout(); plt.show()

## The model

`set_classes(names, get_text_pe(names))` writes the CLIP text embeddings of our
four prompts into the detection head, so the model segments concepts it was
never trained on.

In [ ]:
from ultralytics import YOLOE

t0 = time.time()
model = YOLOE(variant)
model.set_classes(PROMPTS, model.get_text_pe(PROMPTS))
n_par = sum(p.numel() for p in model.model.parameters())
print(f"{variant}   {n_par/1e6:.1f}M params   loaded in {time.time()-t0:.1f}s")
print("prompts:", PROMPTS)

## One patch, one call

Each patch is cut at native resolution and handed to the model whole — no
tiling inside it, so the ground extent in frame is exactly `patch_m`.
`masks.xy` comes back in patch-pixel coordinates; adding the patch origin puts
it back on the orthophoto.

In [ ]:
def run_patch(x0, y0, side_px):
    """Predict on one native-resolution crop. Returns a list of detections."""
    crop = np.array(image.crop((x0, y0, x0 + side_px, y0 + side_px)))
    r = model.predict(crop, imgsz=imgsz, conf=conf, verbose=False)[0]
    out = []
    if r.masks is None:
        return out
    area = float(r.masks.data.shape[-1] * r.masks.data.shape[-2])
    for poly, k, cf, m in zip(r.masks.xy, r.boxes.cls, r.boxes.conf, r.masks.data):
        out.append({
            "cls":  CLASSES[int(k)],
            "conf": float(cf),
            "frac": float(m.sum()) / area,           # share of the patch covered
            "xy":   np.asarray(poly) + np.array([x0, y0]),
        })
    return out


def grid(side_px):
    """Top-left corners of a non-overlapping cover of the orthophoto."""
    return [(x, y)
            for y in range(0, H - side_px + 1, side_px)
            for x in range(0, W - side_px + 1, side_px)]

## Over the whole orthophoto

Non-overlapping patches at the chosen extent, every one of them.

In [ ]:
side = int(round(patch_m / mpp))
cells_xy = grid(side)
print(f"{len(cells_xy)} patches of {patch_m} m ({side} px) — "
      f"about {len(cells_xy)*2.2/60:.1f} min on CPU")

t0 = time.time()
dets, per_patch = [], []
for i, (x0, y0) in enumerate(cells_xy):
    d = run_patch(x0, y0, side)
    dets.extend(d)
    per_patch.append(len(d))
    if (i + 1) % 20 == 0 or i + 1 == len(cells_xy):
        print(f"  {i+1:4d}/{len(cells_xy)}  {len(dets)} dets  {time.time()-t0:5.0f}s")
print(f"done in {time.time()-t0:.0f}s")

## What came back

In [ ]:
n_empty = sum(1 for n in per_patch if n == 0)
print(f"patches: {len(per_patch)}   empty: {n_empty} ({100*n_empty/len(per_patch):.0f}%)   "
      f"detections: {len(dets)}\n")

print(f"{'class':10s} {'dets':>5s} {'patches':>8s} {'mean conf':>10s} "
      f"{'max conf':>9s} {'median frac':>12s} {'>50% frame':>11s}")
for c in CLASSES:
    d = [x for x in dets if x["cls"] == c]
    if not d:
        print(f"{c:10s} {0:5d} {0:8d} {'-':>10s} {'-':>9s} {'-':>12s} {'-':>11s}")
        continue
    cf = np.array([x["conf"] for x in d]); fr = np.array([x["frac"] for x in d])
    npatch = len({(round(x['xy'][:,0].mean()//side), round(x['xy'][:,1].mean()//side)) for x in d})
    print(f"{c:10s} {len(d):5d} {npatch:8d} {cf.mean():10.3f} {cf.max():9.3f} "
          f"{np.median(fr):12.3f} {100*(fr>0.5).mean():10.0f}%")

## The whole scene, with every mask on it

Masks are outlined rather than filled so the imagery underneath stays visible.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 12))
ax.imshow(image)
for d in dets:
    ax.add_patch(Polygon(d["xy"], closed=True, fill=False,
                         edgecolor=COLOUR[d["cls"]], linewidth=1.4, alpha=0.9))
for (x0, y0) in cells_xy:
    ax.add_patch(Polygon([(x0, y0), (x0+side, y0), (x0+side, y0+side), (x0, y0+side)],
                         closed=True, fill=False, edgecolor="white",
                         linewidth=0.3, alpha=0.25))
ax.set_title(f"{variant} — {patch_m} m patches, conf {conf} — {len(dets)} masks")
ax.legend(handles=[Patch(color=COLOUR[c], label=c) for c in CLASSES],
          loc="upper right", framealpha=0.9)
ax.axis("off")
plt.tight_layout()
fig.savefig(FIGURES / "yoloe26-gonchary-overview.png", dpi=110, bbox_inches="tight")
plt.show()

## A sample of individual patches, full resolution

The patches with the most detections, so there is something to look at.

In [ ]:
order = np.argsort(per_patch)[::-1][:n_sample]
ncol  = 4
nrow  = int(np.ceil(len(order) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4*ncol, 4*nrow))
for ax, idx in zip(np.ravel(axes), order):
    x0, y0 = cells_xy[idx]
    ax.imshow(image.crop((x0, y0, x0+side, y0+side)))
    lab = []
    for d in dets:
        cx, cy = d["xy"][:,0].mean(), d["xy"][:,1].mean()
        if x0 <= cx < x0+side and y0 <= cy < y0+side:
            ax.add_patch(Polygon(d["xy"] - np.array([x0, y0]), closed=True, fill=False,
                                 edgecolor=COLOUR[d["cls"]], linewidth=2))
            lab.append(f"{d['cls']} {d['conf']:.2f}")
    ax.set_title(" · ".join(lab[:3]) or "no detections", fontsize=8)
    ax.axis("off")
for ax in np.ravel(axes)[len(order):]:
    ax.axis("off")
fig.suptitle(f"{variant} — {patch_m} m patches at native {mpp*100:.1f} cm/px", y=1.0)
plt.tight_layout()
fig.savefig(FIGURES / "yoloe26-gonchary-patches.png", dpi=110, bbox_inches="tight")
plt.show()

## Does patch size change anything

Same model, same prompts, same four centres — only how much ground is in frame
changes. Every crop is resized to `imgsz`, so this varies apparent object size
and nothing else.

In [ ]:
centres = [(W//4, H//4), (W//2, H//2), (3*W//4, H//2), (W//2, 3*H//4)]

sweep = []
for m_ext in sizes:
    s = int(round(m_ext / mpp))
    for (cx, cy) in centres:
        x0 = int(np.clip(cx - s//2, 0, W - s)); y0 = int(np.clip(cy - s//2, 0, H - s))
        d = run_patch(x0, y0, s)
        sweep.append({"extent": m_ext, "origin": (x0, y0), "side": s, "dets": d})

print(f"{'extent':>7s} {'px':>6s} {'dets':>5s} {'median frac':>12s} {'max conf':>9s}  classes")
for m_ext in sizes:
    rows = [r for r in sweep if r["extent"] == m_ext]
    d = [x for r in rows for x in r["dets"]]
    cnt = Counter(x["cls"] for x in d)
    fr  = [x["frac"] for x in d]; cf = [x["conf"] for x in d]
    print(f"{m_ext:6d}m {rows[0]['side']:6d} {len(d):5d} "
          f"{(np.median(fr) if fr else 0):12.3f} {(max(cf) if cf else 0):9.3f}  {dict(cnt)}")

In [ ]:
fig, axes = plt.subplots(len(sizes), len(centres),
                         figsize=(4*len(centres), 4*len(sizes)))
for i, m_ext in enumerate(sizes):
    rows = [r for r in sweep if r["extent"] == m_ext]
    for j, r in enumerate(rows):
        ax = axes[i][j]
        x0, y0 = r["origin"]; s = r["side"]
        ax.imshow(image.crop((x0, y0, x0+s, y0+s)))
        for d in r["dets"]:
            ax.add_patch(Polygon(d["xy"] - np.array([x0, y0]), closed=True, fill=False,
                                 edgecolor=COLOUR[d["cls"]], linewidth=2))
        ax.set_title(f"{m_ext} m — {len(r['dets'])} masks", fontsize=9)
        ax.axis("off")
fig.suptitle(f"{variant} — same four places at {', '.join(str(s)+' m' for s in sizes)}", y=1.0)
plt.tight_layout()
fig.savefig(FIGURES / "yoloe26-gonchary-sizes.png", dpi=110, bbox_inches="tight")
plt.show()

## Raw results to disk

Polygons dropped — counts, confidences and coverage only.

In [ ]:
out = {
    "variant": variant, "imgsz": imgsz, "conf": conf,
    "image": ORTHO.name, "mpp": mpp, "size_px": [W, H],
    "extent_m": [round(width_m), round(height_m)],
    "patch_m": patch_m, "n_patches": len(cells_xy),
    "detections": [{k: v for k, v in d.items() if k != "xy"} for d in dets],
    "sweep": [{"extent": r["extent"], "origin": r["origin"], "side": r["side"],
               "dets": [{k: v for k, v in d.items() if k != "xy"} for d in r["dets"]]}
              for r in sweep],
}
path = HERE / "results-yoloe26-gonchary.json"
path.write_text(json.dumps(out, indent=1))
print(f"wrote {path.name}  ({path.stat().st_size/1024:.0f} KB)")

---

No verdict. The figures are `figures/yoloe26-gonchary-overview.png`,
`figures/yoloe26-gonchary-patches.png` and `figures/yoloe26-gonchary-sizes.png`.